# PDF Extraction Demo — DataFlow Technical Report

This notebook demonstrates the PDF extraction prototype for the DataFlow Technical Report.

## Flow

```text
PDF file → Load PDF → Extract text page by page → Count pages → Count characters
→ Detect empty pages → Save raw PDF copy → Save staging text/CSV
→ Save clean CSV → Generate PDF metadata → Generate ingestion log
```

## Expected Input

```text
data/sample_inputs/DataFlow_Technical_Report.pdf
```

## Expected Outputs

```text
data/raw/pdf/dataflow_technical_report_raw.pdf
data/staging/pdf/dataflow_pdf_text.txt
data/staging/pdf/dataflow_pdf_pages_staging.csv
data/clean/pdf/dataflow_pdf_pages_clean.csv
logs/pdf_metadata.json
logs/pdf_ingestion_log.json
```

## 1. Import Required Libraries

In [1]:
import json
import uuid
import re
import shutil
import pandas as pd
import pdfplumber

from pathlib import Path
from datetime import datetime, timezone

## 2. Define Project Paths

In [2]:
def find_project_root(current_path: Path) -> Path:
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError("Project root not found. Make sure data/ folder exists.")


def to_relative_path(path: Path, project_root: Path) -> str:
    return path.resolve().relative_to(project_root.resolve()).as_posix()


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

input_path = PROJECT_ROOT / "data" / "sample_inputs" / "DataFlow_Technical_Report.pdf"

raw_dir = PROJECT_ROOT / "data" / "raw" / "pdf"
staging_dir = PROJECT_ROOT / "data" / "staging" / "pdf"
clean_dir = PROJECT_ROOT / "data" / "clean" / "pdf"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "dataflow_technical_report_raw.pdf"
staging_text_output_path = staging_dir / "dataflow_pdf_text.txt"
staging_csv_output_path = staging_dir / "dataflow_pdf_pages_staging.csv"
clean_output_path = clean_dir / "dataflow_pdf_pages_clean.csv"
metadata_output_path = log_dir / "pdf_metadata.json"
log_output_path = log_dir / "pdf_ingestion_log.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Raw output:", raw_output_path)
print("Staging text output:", staging_text_output_path)
print("Staging CSV output:", staging_csv_output_path)
print("Clean output:", clean_output_path)
print("Metadata output:", metadata_output_path)
print("Log output:", log_output_path)

Current dir: f:\data\new\quanskill\DataVision_Duy\week2\notebooks\data_team
Project root: F:\data\new\quanskill\DataVision_Duy\week2
Input path: F:\data\new\quanskill\DataVision_Duy\week2\data\sample_inputs\DataFlow_Technical_Report.pdf
Input exists: True
Raw output: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\pdf\dataflow_technical_report_raw.pdf
Staging text output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\pdf\dataflow_pdf_text.txt
Staging CSV output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\pdf\dataflow_pdf_pages_staging.csv
Clean output: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\pdf\dataflow_pdf_pages_clean.csv
Metadata output: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_metadata.json
Log output: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_ingestion_log.json


## 3. Validate Input PDF

In [3]:
if not input_path.exists():
    raise FileNotFoundError(f"PDF file not found: {input_path}")

if input_path.stat().st_size == 0:
    raise ValueError("PDF file is empty.")

if input_path.suffix.lower() != ".pdf":
    raise ValueError("Input file is not a PDF file.")

print("Input PDF validation passed.")
print("File size MB:", round(input_path.stat().st_size / (1024 * 1024), 2))

Input PDF validation passed.
File size MB: 2.73


## 4. Helper Functions

In [4]:
def clean_extracted_text(text: str) -> str:
    if text is None:
        return ""

    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


def count_words(text: str) -> int:
    if not text:
        return 0

    return len(text.split())

## 5. Start Ingestion Run

In [5]:
run_id = str(uuid.uuid4())
source_name = "dataflow_technical_report_pdf"
source_type = "pdf"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: abdc086b-cdf1-433e-9993-5306b811bf53
Start time: 2026-06-14T16:44:52.272730+00:00


## 6. Save Raw PDF Copy

In [6]:
shutil.copy2(input_path, raw_output_path)

print("Raw PDF saved:", to_relative_path(raw_output_path, PROJECT_ROOT))

Raw PDF saved: data/raw/pdf/dataflow_technical_report_raw.pdf


## 7. Extract Text Page by Page

In [7]:
page_records = []

try:
    with pdfplumber.open(input_path) as pdf:
        total_pages = len(pdf.pages)

        for page_index, page in enumerate(pdf.pages, start=1):
            raw_text = page.extract_text() or ""
            clean_text = clean_extracted_text(raw_text)

            page_records.append(
                {
                    "page_number": page_index,
                    "raw_text": raw_text,
                    "clean_text": clean_text,
                    "char_count": len(clean_text),
                    "word_count": count_words(clean_text),
                    "is_empty_page": len(clean_text) == 0,
                }
            )

    status = "success"
    error_message = None

    print("PDF extracted successfully.")
    print("Total pages:", total_pages)
    print("Pages extracted:", len(page_records))

except Exception as error:
    status = "failed"
    error_message = str(error)
    raise

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


PDF extracted successfully.
Total pages: 36
Pages extracted: 36


## 8. Convert Extraction Result to DataFrame

In [8]:
df_pages = pd.DataFrame(page_records)

records_read = len(df_pages)
empty_page_count = int(df_pages["is_empty_page"].sum())
total_characters = int(df_pages["char_count"].sum())
total_words = int(df_pages["word_count"].sum())

print("Records read:", records_read)
print("Empty pages:", empty_page_count)
print("Total characters:", total_characters)
print("Total words:", total_words)

df_pages.head(5)

Records read: 36
Empty pages: 0
Total characters: 129028
Total words: 17536


,page_number,raw_text,clean_text,char_count,word_count,is_empty_page
0,1,"December19,2025\nDataFlow: An LLM-Driven Frame...","December19,2025 DataFlow: An LLM-Driven Framew...",2953,343,False
1,2,Contents\n1 Introduction . . . . . . . . . . ....,Contents 1 Introduction . . . . . . . . . . . ...,4920,1922,False
2,3,DataFlow Technical Report 3\n7.7.1 Experimenta...,DataFlow Technical Report 3 7.7.1 Experimental...,769,305,False
3,4,DataFlow Technical Report 4\n1 Introduction\nL...,DataFlow Technical Report 4 1 Introduction Lar...,4914,558,False
4,5,DataFlow Technical Report 5\nTo ensure usabili...,DataFlow Technical Report 5 To ensure usabilit...,4461,488,False


## 9. Save Staging Text File

In [9]:
combined_text = "\n\n".join(
    [
        f"--- Page {row['page_number']} ---\n{row['clean_text']}"
        for _, row in df_pages.iterrows()
    ]
)

with open(staging_text_output_path, "w", encoding="utf-8") as file:
    file.write(combined_text)

print("Staging text saved:", to_relative_path(staging_text_output_path, PROJECT_ROOT))

Staging text saved: data/staging/pdf/dataflow_pdf_text.txt


## 10. Save Staging CSV

In [10]:
df_pages.to_csv(staging_csv_output_path, index=False, encoding="utf-8")
print("Staging CSV saved:", to_relative_path(staging_csv_output_path, PROJECT_ROOT))

Staging CSV saved: data/staging/pdf/dataflow_pdf_pages_staging.csv


## 11. Create Clean PDF Data

In [11]:
df_clean = df_pages.copy()
df_clean = df_clean[df_clean["is_empty_page"] == False].copy()
df_clean = df_clean.drop_duplicates(subset=["clean_text"])

records_valid = len(df_clean)
records_invalid = records_read - records_valid

print("Records read:", records_read)
print("Records valid:", records_valid)
print("Records invalid:", records_invalid)

df_clean.head(5)

Records read: 36
Records valid: 36
Records invalid: 0


,page_number,raw_text,clean_text,char_count,word_count,is_empty_page
0,1,"December19,2025\nDataFlow: An LLM-Driven Frame...","December19,2025 DataFlow: An LLM-Driven Framew...",2953,343,False
1,2,Contents\n1 Introduction . . . . . . . . . . ....,Contents 1 Introduction . . . . . . . . . . . ...,4920,1922,False
2,3,DataFlow Technical Report 3\n7.7.1 Experimenta...,DataFlow Technical Report 3 7.7.1 Experimental...,769,305,False
3,4,DataFlow Technical Report 4\n1 Introduction\nL...,DataFlow Technical Report 4 1 Introduction Lar...,4914,558,False
4,5,DataFlow Technical Report 5\nTo ensure usabili...,DataFlow Technical Report 5 To ensure usabilit...,4461,488,False


## 12. Save Clean Output

In [12]:
df_clean.to_csv(clean_output_path, index=False, encoding="utf-8")
print("Clean CSV saved:", to_relative_path(clean_output_path, PROJECT_ROOT))

Clean CSV saved: data/clean/pdf/dataflow_pdf_pages_clean.csv


## 13. Generate PDF Metadata

In [13]:
pdf_metadata = {
    "file_name": input_path.name,
    "file_size_bytes": input_path.stat().st_size,
    "file_size_mb": round(input_path.stat().st_size / (1024 * 1024), 2),
    "total_pages": int(total_pages),
    "pages_extracted": int(records_read),
    "empty_page_count": int(empty_page_count),
    "total_characters": int(total_characters),
    "total_words": int(total_words),
    "raw_output_path": to_relative_path(raw_output_path, PROJECT_ROOT),
    "staging_text_output_path": to_relative_path(staging_text_output_path, PROJECT_ROOT),
    "staging_csv_output_path": to_relative_path(staging_csv_output_path, PROJECT_ROOT),
    "clean_output_path": to_relative_path(clean_output_path, PROJECT_ROOT),
}

with open(metadata_output_path, "w", encoding="utf-8") as file:
    json.dump(pdf_metadata, file, indent=4, ensure_ascii=False)

print("PDF metadata saved:", to_relative_path(metadata_output_path, PROJECT_ROOT))
pdf_metadata

PDF metadata saved: logs/pdf_metadata.json


{'file_name': 'DataFlow_Technical_Report.pdf',
 'file_size_bytes': 2857707,
 'file_size_mb': 2.73,
 'total_pages': 36,
 'pages_extracted': 36,
 'empty_page_count': 0,
 'total_characters': 129028,
 'total_words': 17536,
 'raw_output_path': 'data/raw/pdf/dataflow_technical_report_raw.pdf',
 'staging_text_output_path': 'data/staging/pdf/dataflow_pdf_text.txt',
 'staging_csv_output_path': 'data/staging/pdf/dataflow_pdf_pages_staging.csv',
 'clean_output_path': 'data/clean/pdf/dataflow_pdf_pages_clean.csv'}

## 14. Generate Ingestion Log

In [14]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": to_relative_path(input_path, PROJECT_ROOT),
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "error_message": error_message,
    "raw_output_path": to_relative_path(raw_output_path, PROJECT_ROOT),
    "staging_output_path": to_relative_path(staging_csv_output_path, PROJECT_ROOT),
    "clean_output_path": to_relative_path(clean_output_path, PROJECT_ROOT),
    "metadata_output_path": to_relative_path(metadata_output_path, PROJECT_ROOT),
    "owner": owner,
    "pdf_metadata": pdf_metadata,
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("PDF ingestion log saved:", to_relative_path(log_output_path, PROJECT_ROOT))
ingestion_log

PDF ingestion log saved: logs/pdf_ingestion_log.json


{'run_id': 'abdc086b-cdf1-433e-9993-5306b811bf53',
 'source_name': 'dataflow_technical_report_pdf',
 'source_type': 'pdf',
 'input_path_or_url': 'data/sample_inputs/DataFlow_Technical_Report.pdf',
 'start_time': '2026-06-14T16:44:52.272730+00:00',
 'end_time': '2026-06-14T16:45:13.967593+00:00',
 'status': 'success',
 'records_read': 36,
 'records_valid': 36,
 'records_invalid': 0,
 'error_message': None,
 'raw_output_path': 'data/raw/pdf/dataflow_technical_report_raw.pdf',
 'staging_output_path': 'data/staging/pdf/dataflow_pdf_pages_staging.csv',
 'clean_output_path': 'data/clean/pdf/dataflow_pdf_pages_clean.csv',
 'metadata_output_path': 'logs/pdf_metadata.json',
 'owner': 'Nguyen Minh Duy',
 'pdf_metadata': {'file_name': 'DataFlow_Technical_Report.pdf',
  'file_size_bytes': 2857707,
  'file_size_mb': 2.73,
  'total_pages': 36,
  'pages_extracted': 36,
  'empty_page_count': 0,
  'total_characters': 129028,
  'total_words': 17536,
  'raw_output_path': 'data/raw/pdf/dataflow_technical_

## 15. Final Output Check

In [15]:
print("Raw exists:", raw_output_path.exists())
print("Staging text exists:", staging_text_output_path.exists())
print("Staging CSV exists:", staging_csv_output_path.exists())
print("Clean exists:", clean_output_path.exists())
print("Metadata exists:", metadata_output_path.exists())
print("Log exists:", log_output_path.exists())

print("\nOutput files:")
print(to_relative_path(raw_output_path, PROJECT_ROOT))
print(to_relative_path(staging_text_output_path, PROJECT_ROOT))
print(to_relative_path(staging_csv_output_path, PROJECT_ROOT))
print(to_relative_path(clean_output_path, PROJECT_ROOT))
print(to_relative_path(metadata_output_path, PROJECT_ROOT))
print(to_relative_path(log_output_path, PROJECT_ROOT))

Raw exists: True
Staging text exists: True
Staging CSV exists: True
Clean exists: True
Metadata exists: True
Log exists: True

Output files:
data/raw/pdf/dataflow_technical_report_raw.pdf
data/staging/pdf/dataflow_pdf_text.txt
data/staging/pdf/dataflow_pdf_pages_staging.csv
data/clean/pdf/dataflow_pdf_pages_clean.csv
logs/pdf_metadata.json
logs/pdf_ingestion_log.json


## 16. Summary

```text
data/raw/pdf/dataflow_technical_report_raw.pdf
data/staging/pdf/dataflow_pdf_text.txt
data/staging/pdf/dataflow_pdf_pages_staging.csv
data/clean/pdf/dataflow_pdf_pages_clean.csv
logs/pdf_metadata.json
logs/pdf_ingestion_log.json
```